## 1. Configuration and Preprocessing

In [4]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================

import os
import json
import numpy as np


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

METADATA_FILE = (
    "../fma_data_preparation/data/"
    "dataset1000/metadata1000.json"
)

OUTPUT_DIR = (
    "../fma_data_preparation/data/"
    "fma_dataset/fma_files_watermarked"
)


# ------------------------------------------------------------
# Watermarking parameters
# ------------------------------------------------------------

REP_CODE = True
FRAME_LENGTH = 4096
CONTROL_STRENGTH = 0.1
OVERLAP = 0.0
NUM_REPS = 5


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


# ------------------------------------------------------------
# Prepare output directory
# ------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

if not os.path.isfile(METADATA_FILE):
    raise FileNotFoundError(
        f"Metadata file not found:\n"
        f"{os.path.abspath(METADATA_FILE)}"
    )

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)


if not isinstance(metadata, list):
    raise ValueError(
        "metadata1000.json must contain a JSON array."
    )


print("Configuration loaded successfully.")
print(f"Metadata records : {len(metadata)}")
print(f"Output directory  : {os.path.abspath(OUTPUT_DIR)}")
print(f"Frame length      : {FRAME_LENGTH}")
print(f"Strength          : {CONTROL_STRENGTH}")
print(f"Repetition coding : {REP_CODE}")
print(f"Repetitions       : {NUM_REPS}")

Configuration loaded successfully.
Metadata records : 1000
Output directory  : c:\Users\Ideapad\codesk\KABU\fma_data_preparation\data\fma_dataset\fma_files_watermarked
Frame length      : 4096
Strength          : 0.1
Repetition coding : True
Repetitions       : 5


## 2. Watermark Embedding

In [5]:
# ============================================================
# CELL 2 — BATCH WATERMARK EMBEDDING
# ============================================================

from scipy.io import wavfile
from pydub import AudioSegment


# ------------------------------------------------------------
# Helper: text -> binary
# ------------------------------------------------------------

def text_to_binary(text):
    return "".join(
        format(ord(char), "08b")
        for char in text
    )


# ------------------------------------------------------------
# Generate deterministic pseudo-random sequence
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

prs = rng.random(FRAME_LENGTH) - 0.5


# ------------------------------------------------------------
# Watermark a single audio file
# ------------------------------------------------------------

def watermark_audio(input_file, output_file, watermark_text):
    """
    Embed watermark_text into an audio file using the
    configured PRS watermarking parameters.
    """

    # --------------------------------------------------------
    # Convert source MP3 to mono WAV in memory
    # --------------------------------------------------------

    audio = AudioSegment.from_file(input_file)
    audio = audio.set_channels(1)

    samples = np.array(
        audio.get_array_of_samples(),
        dtype=np.float64
    )

    sample_width = audio.sample_width
    sample_rate = audio.frame_rate

    # --------------------------------------------------------
    # Determine signal range
    # --------------------------------------------------------

    if sample_width == 1:
        max_amplitude = 127
    elif sample_width == 2:
        max_amplitude = 32767
    elif sample_width == 4:
        max_amplitude = 2147483647
    else:
        raise ValueError(
            f"Unsupported sample width: {sample_width}"
        )

    signal_len = len(samples)

    frame_shift = int(
        FRAME_LENGTH * (1 - OVERLAP)
    )

    overlap_length = int(
        FRAME_LENGTH * OVERLAP
    )

    embed_nbit = int(
        np.floor(
            (signal_len - overlap_length)
            / frame_shift
        )
    )

    # --------------------------------------------------------
    # Repetition coding
    # --------------------------------------------------------

    if REP_CODE:
        effective_nbit = embed_nbit // NUM_REPS
        embed_nbit = effective_nbit * NUM_REPS
    else:
        effective_nbit = embed_nbit

    if effective_nbit <= 0:
        raise ValueError(
            "Audio is too short for the configured "
            "FRAME_LENGTH and NUM_REPS."
        )

    # --------------------------------------------------------
    # Convert artist ID to binary
    # --------------------------------------------------------

    binary_str = text_to_binary(watermark_text)

    # Ensure the watermark fits the available capacity
    if len(binary_str) > effective_nbit:
        raise ValueError(
            f"Watermark '{watermark_text}' requires "
            f"{len(binary_str)} bits, but only "
            f"{effective_nbit} bits are available."
        )

    # Pad remaining capacity with zeros
    binary_str = binary_str.ljust(
        effective_nbit,
        "0"
    )

    wmark_original = np.array(
        [int(bit) for bit in binary_str],
        dtype=np.int8
    )

    # --------------------------------------------------------
    # Apply repetition coding
    # --------------------------------------------------------

    if REP_CODE:
        wmark_extended = np.repeat(
            wmark_original,
            NUM_REPS
        )
    else:
        wmark_extended = wmark_original

    # --------------------------------------------------------
    # Embed watermark frame-by-frame
    # --------------------------------------------------------

    wmarked_signal = np.zeros(
        frame_shift * embed_nbit,
        dtype=np.float64
    )

    pointer = 0

    for i in range(embed_nbit):

        frame = samples[
            pointer:pointer + FRAME_LENGTH
        ].copy()

        if len(frame) < FRAME_LENGTH:
            break

        alpha = (
            CONTROL_STRENGTH
            * np.max(np.abs(frame))
        )

        if wmark_extended[i] == 1:
            frame += alpha * prs
        else:
            frame -= alpha * prs

        start = i * frame_shift
        end = start + frame_shift

        wmarked_signal[start:end] = (
            frame[:frame_shift]
        )

        pointer += frame_shift

    # --------------------------------------------------------
    # Preserve remaining audio
    # --------------------------------------------------------

    if len(wmarked_signal) < signal_len:

        wmarked_signal = np.concatenate(
            (
                wmarked_signal,
                samples[len(wmarked_signal):]
            )
        )

    # --------------------------------------------------------
    # Clip and convert to 16-bit PCM
    # --------------------------------------------------------

    wmarked_signal = np.clip(
        wmarked_signal,
        -32768,
        32767
    ).astype(np.int16)

    # --------------------------------------------------------
    # Save watermarked WAV
    # --------------------------------------------------------

    wavfile.write(
        output_file,
        sample_rate,
        wmarked_signal
    )

    return {
        "watermark_text": watermark_text,
        "watermark_bits": int(len(binary_str)),
        "effective_bits": int(effective_nbit),
        "sample_rate": int(sample_rate),
        "duration_seconds": float(
            signal_len / sample_rate
        )
    }


# ============================================================
# PROCESS DATASET
# ============================================================

processing_results = []

print("=" * 60)
print("STARTING BATCH WATERMARK EMBEDDING")
print("=" * 60)

for index, record in enumerate(metadata, start=1):

    track_id = str(
        record["custom_track_id"]
    )

    watermark_text = str(
        record["custom_artist_id"]
    )

    input_file = record["file"]

    output_file = os.path.join(
        OUTPUT_DIR,
        f"{track_id}_watermarked.wav"
    )

    try:

        if not os.path.isfile(input_file):
            raise FileNotFoundError(
                f"Source file not found: {input_file}"
            )

        result = watermark_audio(
            input_file=input_file,
            output_file=output_file,
            watermark_text=watermark_text
        )

        processing_results.append({
            "custom_track_id": track_id,
            "custom_artist_id": watermark_text,
            "input_file": input_file,
            "output_file": output_file,
            "status": "success",
            **result
        })

        print(
            f"[{index}/{len(metadata)}] "
            f"✓ {track_id} → {watermark_text}"
        )

    except Exception as error:

        processing_results.append({
            "custom_track_id": track_id,
            "custom_artist_id": watermark_text,
            "input_file": input_file,
            "output_file": output_file,
            "status": "failed",
            "error": str(error)
        })

        print(
            f"[{index}/{len(metadata)}] "
            f"✗ {track_id}: {error}"
        )


# ============================================================
# SUMMARY
# ============================================================

successful = sum(
    result["status"] == "success"
    for result in processing_results
)

failed = sum(
    result["status"] == "failed"
    for result in processing_results
)

print("\n" + "=" * 60)
print("WATERMARKING COMPLETE")
print("=" * 60)

print(f"Total records : {len(metadata)}")
print(f"Successful    : {successful}")
print(f"Failed        : {failed}")
print(f"Output folder : {os.path.abspath(OUTPUT_DIR)}")

STARTING BATCH WATERMARK EMBEDDING
[1/1000] ✓ 5239 → Emin19
[2/1000] ✓ 0913 → Daft05
[3/1000] ✓ 0205 → Just08
[4/1000] ✓ 6075 → Dual10
[5/1000] ✓ 2254 → Tayl03
[6/1000] ✓ 2007 → Drak16
[7/1000] ✓ 1829 → Lady06
[8/1000] ✓ 1144 → Drak16
[9/1000] ✓ 6034 → Beyo01
[10/1000] ✓ 0840 → Aria14
[11/1000] ✓ 5544 → Tayl03
[12/1000] ✓ 6068 → Dual10
[13/1000] ✓ 7309 → Just08
[14/1000] ✓ 4468 → Lady06
[15/1000] ✓ 0713 → Edsh11
[16/1000] ✓ 4838 → Drak16
[17/1000] ✓ 3457 → Daft05
[18/1000] ✓ 0261 → Emin19
[19/1000] ✓ 0245 → Cold18
[20/1000] ✓ 0768 → Daft05
[21/1000] ✓ 1792 → Lady06
[22/1000] ✓ 1906 → Kany07
[23/1000] ✓ 4140 → Beyo01
[24/1000] ✓ 4932 → Emin19
[25/1000] ✓ 0218 → Lady06
[26/1000] ✓ 4598 → Beyo01
[27/1000] ✓ 1629 → Emin19
[28/1000] ✓ 5866 → Edsh11
[29/1000] ✓ 5324 → Just08
[30/1000] ✓ 5746 → Daft05
[31/1000] ✓ 4465 → Adel20
[32/1000] ✓ 3437 → Kend15
[33/1000] ✓ 1806 → Lady06
[34/1000] ✓ 3680 → Dual10
[35/1000] ✓ 4828 → Dual10
[36/1000] ✓ 2279 → Emin19
[37/1000] ✓ 6631 → Dual10
[38/1000] ✓ 

## 3. Detection, Decoding and Evaluation

In [6]:
import time


# ------------------------------------------------------------
# Blind detection for one watermarked audio file
# ------------------------------------------------------------

def blind_detect(
    audio_file,
    watermark_text,
    prs
):
    """
    Blindly detects a watermark without access to the
    original host audio signal.
    """

    start_time = time.perf_counter()

    _, eval_signal = wavfile.read(audio_file)

    if eval_signal.ndim > 1:
        eval_signal = eval_signal[:, 0]

    eval_signal = eval_signal.astype(np.float64)

    signal_len = len(eval_signal)

    frame_shift = int(
        FRAME_LENGTH * (1 - OVERLAP)
    )

    embed_nbit = int(
        np.floor(
            (
                signal_len
                - int(FRAME_LENGTH * OVERLAP)
            )
            / frame_shift
        )
    )

    # --------------------------------------------------------
    # Determine recoverable watermark bits
    # --------------------------------------------------------

    if REP_CODE:
        effective_nbit = embed_nbit // NUM_REPS
        embed_nbit = effective_nbit * NUM_REPS
    else:
        effective_nbit = embed_nbit

    # --------------------------------------------------------
    # Expected watermark
    # --------------------------------------------------------

    expected_binary = text_to_binary(
        watermark_text
    )

    if len(expected_binary) > effective_nbit:
        expected_binary = expected_binary[
            :effective_nbit
        ]
    else:
        expected_binary = expected_binary.ljust(
            effective_nbit,
            "0"
        )

    expected_bits = np.array(
        [int(bit) for bit in expected_binary],
        dtype=np.int8
    )

    # --------------------------------------------------------
    # Blind correlation detection
    # --------------------------------------------------------

    detected_bit = np.zeros(
        embed_nbit,
        dtype=np.int8
    )

    pointer = 0

    for i in range(embed_nbit):

        frame = eval_signal[
            pointer:pointer + FRAME_LENGTH
        ]

        if len(frame) < FRAME_LENGTH:
            break

        correlation = np.correlate(
            frame,
            prs,
            mode="full"
        )

        peak = np.argmax(
            np.abs(correlation)
        )

        detected_bit[i] = (
            1
            if correlation[peak] >= 0
            else 0
        )

        pointer += frame_shift

    # --------------------------------------------------------
    # Repetition-code recovery
    # --------------------------------------------------------

    if REP_CODE:

        recovered_bits = np.zeros(
            effective_nbit,
            dtype=np.int8
        )

        for i in range(effective_nbit):

            start = i * NUM_REPS
            end = start + NUM_REPS

            chunk = detected_bit[start:end]

            recovered_bits[i] = (
                1
                if np.mean(chunk) >= 0.5
                else 0
            )

    else:

        recovered_bits = detected_bit[
            :effective_nbit
        ]

    # --------------------------------------------------------
    # Calculate BER
    # --------------------------------------------------------

    comparable_length = min(
        len(expected_bits),
        len(recovered_bits)
    )

    if comparable_length > 0:

        bit_errors = int(
            np.sum(
                expected_bits[:comparable_length]
                != recovered_bits[:comparable_length]
            )
        )

        ber = (
            bit_errors
            / comparable_length
            * 100
        )

    else:

        bit_errors = 0
        ber = 100.0

    # --------------------------------------------------------
    # Decode recovered binary
    # --------------------------------------------------------

    recovered_binary = "".join(
        str(int(bit))
        for bit in recovered_bits
    )

    chars = [
        recovered_binary[i:i + 8]
        for i in range(
            0,
            len(recovered_binary),
            8
        )
    ]

    recovered_text = "".join(
        chr(int(char, 2))
        for char in chars
        if len(char) == 8
        and char != "00000000"
    )

    # --------------------------------------------------------
    # Detection result
    # --------------------------------------------------------

    detection_success = (
        recovered_text == watermark_text
    )

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    return {
        "expected_watermark": watermark_text,
        "recovered_watermark": recovered_text,
        "expected_bits": int(len(expected_bits)),
        "recovered_bits": int(len(recovered_bits)),
        "bit_errors": bit_errors,
        "ber_percent": round(ber, 4),
        "detection_success": bool(
            detection_success
        ),
        "processing_time_seconds": round(
            elapsed_time,
            6
        )
    }


# ============================================================
# BATCH BLIND DETECTION
# ============================================================

detection_results = []

print("=" * 60)
print("STARTING BLIND WATERMARK DETECTION")
print("=" * 60)

for index, record in enumerate(
    processing_results,
    start=1
):

    # Skip files that failed during embedding
    if record["status"] != "success":
        continue

    track_id = record[
        "custom_track_id"
    ]

    artist_id = record[
        "custom_artist_id"
    ]

    watermarked_file = record[
        "output_file"
    ]

    try:

        result = blind_detect(
            audio_file=watermarked_file,
            watermark_text=artist_id,
            prs=prs
        )

        detection_results.append({
            "custom_track_id": track_id,
            "custom_artist_id": artist_id,
            "input_file": record[
                "input_file"
            ],
            "watermarked_file": watermarked_file,
            "status": "success",
            **result
        })

        status_symbol = (
            "✓"
            if result["detection_success"]
            else "✗"
        )

        print(
            f"[{index}/{len(processing_results)}] "
            f"{status_symbol} {track_id} → "
            f"{result['recovered_watermark']} "
            f"(BER: {result['ber_percent']:.2f}%)"
        )

    except Exception as error:

        detection_results.append({
            "custom_track_id": track_id,
            "custom_artist_id": artist_id,
            "input_file": record[
                "input_file"
            ],
            "watermarked_file": watermarked_file,
            "status": "failed",
            "error": str(error)
        })

        print(
            f"[{index}/{len(processing_results)}] "
            f"✗ {track_id}: {error}"
        )


# ============================================================
# SAVE RESULTS
# ============================================================

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "blind_detection_results.json"
)

with open(
    RESULTS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        detection_results,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# SUMMARY
# ============================================================

successful_detections = [
    r for r in detection_results
    if r["status"] == "success"
]

correct = sum(
    r["detection_success"]
    for r in successful_detections
)

total = len(successful_detections)

accuracy = (
    correct / total * 100
    if total > 0
    else 0
)

print("\n" + "=" * 60)
print("BLIND DETECTION COMPLETE")
print("=" * 60)

print(f"Files tested       : {total}")
print(f"Correct detections : {correct}")
print(f"Failed detections  : {total - correct}")
print(f"Detection accuracy : {accuracy:.2f}%")
print(f"Results saved to   : {RESULTS_FILE}")

STARTING BLIND WATERMARK DETECTION
[1/1000] ✓ 5239 → Emin19 (BER: 0.00%)
[2/1000] ✓ 0913 → Daft05 (BER: 0.00%)
[3/1000] ✓ 0205 → Just08 (BER: 0.00%)
[4/1000] ✓ 6075 → Dual10 (BER: 0.00%)
[5/1000] ✓ 2254 → Tayl03 (BER: 0.00%)
[6/1000] ✗ 2007 → h;õï1Jm (BER: 34.38%)
[7/1000] ✓ 1829 → Lady06 (BER: 0.00%)
[8/1000] ✓ 1144 → Drak16 (BER: 0.00%)
[9/1000] ✓ 6034 → Beyo01 (BER: 0.00%)
[10/1000] ✓ 0840 → Aria14 (BER: 0.00%)
[11/1000] ✓ 5544 → Tayl03 (BER: 0.00%)
[12/1000] ✓ 6068 → Dual10 (BER: 0.00%)
[13/1000] ✓ 7309 → Just08 (BER: 0.00%)
[14/1000] ✓ 4468 → Lady06 (BER: 0.00%)
[15/1000] ✓ 0713 → Edsh11 (BER: 0.00%)
[16/1000] ✓ 4838 → Drak16 (BER: 0.00%)
[17/1000] ✓ 3457 → Daft05 (BER: 0.00%)
[18/1000] ✓ 0261 → Emin19 (BER: 0.00%)
[19/1000] ✓ 0245 → Cold18 (BER: 0.00%)
[20/1000] ✓ 0768 → Daft05 (BER: 0.00%)
[21/1000] ✓ 1792 → Lady06 (BER: 0.00%)
[22/1000] ✓ 1906 → Kany07 (BER: 0.00%)
[23/1000] ✓ 4140 → Beyo01 (BER: 0.00%)
[24/1000] ✓ 4932 → Emin19 (BER: 0.00%)
[25/1000] ✓ 0218 → Lady06 (BER: 0.0